# 04 — Baseline Model + Evaluation

Run SeisBench's pretrained PhaseNet on the gold-layer test split, and report
the evaluation this project's standards require:
- Precision/recall/F1 **per event type**, not just overall accuracy
- False-positive rate specifically (noisy alerts get disabled by users)
- Confidence calibration (does a stated 90% confidence mean ~90% real accuracy?)

As noted in `gold_label_split.py`, the `vehicle_human` class has no labeled
source in STEAD/INSTANCE yet — report this eval honestly as effectively a
3-class problem for now, not silently as if all 4 classes were trained.

In [ ]:
import sys
sys.path.insert(0, "../../src/02_ml_pipeline")
import pandas as pd
import replay_pipeline as rp

gold = pd.read_csv("../../data/module2_vibration/gold/stead_gold.csv")
test_set = gold[gold["split"] == "test"]
print(f"{len(test_set)} windows in the held-out test split")

In [ ]:
# Run inference over the test split using the exact same classify_window()
# replay_pipeline.py uses in production, rather than a second, potentially
# drifting implementation here.
#
# An earlier version of this cell called `model.classify(window)` and read
# `event_type`/`confidence` attributes off the result. That was a guess,
# verified wrong against a real seisbench==0.7.0 install (the version
# pinned in requirements.txt): `classify()` expects an obspy.Stream, not a
# raw array, and returns a ClassifyOutput exposing `.picks` (arrival
# picks), not those attributes. See replay_pipeline.py's classify_window()
# docstring for the full verified rationale behind calling PhaseNet's
# forward pass directly instead.
#
# Left as a per-window loop (rather than batched) for readability at
# prototype scale; batch it once the sample size grows beyond a quick demo.
y_true, y_pred, confidences = [], [], []
for _, row in test_set.iterrows():
    result = rp.classify_window("stead", row)
    y_true.append(row["event_type"])
    y_pred.append(result["event_type"])
    confidences.append(result["confidence"])

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_true, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

# False-positive rate: non-seismic windows incorrectly classified as seismic.
non_seismic_mask = [t != "seismic" for t in y_true]
false_positives = sum(
    1 for t, p, m in zip(y_true, y_pred, non_seismic_mask) if m and p == "seismic"
)
n_non_seismic = sum(non_seismic_mask)
fpr = false_positives / n_non_seismic if n_non_seismic else float("nan")
print(f"False-positive rate (non-seismic misclassified as seismic): {fpr:.3f}")

In [ ]:
# Confidence calibration: bucket predictions by stated confidence, check
# whether real accuracy in each bucket roughly matches the stated confidence.
calib_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "confidence": confidences})
calib_df["correct"] = calib_df["y_true"] == calib_df["y_pred"]
calib_df["confidence_bucket"] = pd.cut(calib_df["confidence"], bins=[0, 0.5, 0.7, 0.8, 0.9, 1.0])
print(calib_df.groupby("confidence_bucket")["correct"].mean())

In [ ]:
import model_registry

eval_metrics = {
    "classification_report": classification_report(y_true, y_pred, zero_division=0, output_dict=True),
    "false_positive_rate": fpr,
    "n_test_windows": len(test_set),
    "note": "vehicle_human class untrained — no labeled source in STEAD/INSTANCE",
}
metadata_path = model_registry.write_model_metadata(
    model_name="phasenet",
    model_version="v1",
    data_version="stead_v1",
    eval_metrics=eval_metrics,
)
print(metadata_path)